In [27]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

In [28]:
data = pd.read_csv("digit-recognizer/train.csv")

In [29]:
data.sample(5)

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
11410,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
20925,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9763,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
24454,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
23135,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [30]:
data.describe

<bound method NDFrame.describe of        label  pixel0  pixel1  pixel2  pixel3  pixel4  pixel5  pixel6  pixel7  \
0          1       0       0       0       0       0       0       0       0   
1          0       0       0       0       0       0       0       0       0   
2          1       0       0       0       0       0       0       0       0   
3          4       0       0       0       0       0       0       0       0   
4          0       0       0       0       0       0       0       0       0   
...      ...     ...     ...     ...     ...     ...     ...     ...     ...   
41995      0       0       0       0       0       0       0       0       0   
41996      1       0       0       0       0       0       0       0       0   
41997      7       0       0       0       0       0       0       0       0   
41998      6       0       0       0       0       0       0       0       0   
41999      9       0       0       0       0       0       0       0       0   

     

In [31]:
data = np.array(data)
m, n = data.shape
np.random.shuffle(data)

data_dev = data[0:1000].T
Y_dev = data_dev[0]
X_dev = data_dev[1:n]

data_train = data[1000:m].T
Y_train = data_train[0]
X_train = data_train[1:n]

In [32]:
X_train = X_train.astype(np.float32) / 255.0
X_dev = X_dev.astype(np.float32) / 255.0

In [ ]:
def inti_params():
    W1 = np.random.randn(64, 784) * np.sqrt(2/784)
    b1 = np.zeros((64,1))

    W2 = np.random.randn(10, 64) * np.sqrt(2/64)
    b2 = np.zeros((10,1))
    
    return W1, b1, W2, b2

'''
def inti_params():
    W1 = np.random.rand(10, 784) - 0.5
    b1 = np.random.rand(10, 1) - 0.5
    W2 = np.random.rand(10, 10) - 0.5
    b2 = np.random.rand(10, 1) - 0.5
    return W1, b1, W2, b2
'''

def ReLU(Z):
    return np.maximum(0, Z)

def softmax(Z):
    Z_shifted = Z - np.max(Z, axis=0, keepdims=True)
    exp_Z = np.exp(Z_shifted)
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

def foward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    return one_hot_Y.T

def deriv_ReLU(Z):
    return Z > 0

def back_prop(Z1, A1, Z2, A2, W2, Y, X):
    m = Y.shape[0]

    one_hot_Y = one_hot(Y)
    
    dZ2 = A2 - one_hot_Y
    dW2 = (dZ2.dot(A1.T)) / m
    db2 = np.sum(dZ2, axis=1, keepdims=True) / m
    
    dZ1 = W2.T.dot(dZ2) * deriv_ReLU(Z1)
    dW1 = (dZ1.dot(X.T)) / m
    db1 = np.sum(dZ1, axis=1, keepdims=True) / m
    
    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, learning_rate):
    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1
    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2
    return W1, b1, W2, b2

def loss(X, Y, A2):
    m = Y.size
    log_likelihood = -np.log(A2[Y, range(m)])
    loss = np.sum(log_likelihood) / m
    return loss

def get_prediction(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    print(predictions, Y)
    return np.sum(predictions == Y) / Y.size

def gradient_descent(X, Y, iterations, learning_rate):
    W1, b1, W2, b2 = inti_params()
    for i in range(iterations):
        Z1, A1, Z2, A2 = foward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2, db2 = back_prop(Z1, A1, Z2, A2, W2, Y, X)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, learning_rate)
        
        if (i % 10 == 0):
            print("Itreation: ", i)
            print("Loss: ", loss(X, Y, A2))
            print("Accuracy: ", get_accuracy(get_prediction(A2), Y))
            print("------------------------------------------------------------------------------------------------------")
    
    return W1, b1, W2, b2

In [ ]:
W1, b1, W2, b2 = gradient_descent(X_train, Y_train, iterations=500, learning_rate=0.2)

Itreation:  0
Loss:  2.498423517691266
[6 6 3 ... 6 6 9] [3 5 7 ... 8 5 2]
Accuracy:  0.0795609756097561
------------------------------------------------------------------------------------------------------
Itreation:  10
Loss:  1.1621286880774946
[3 2 7 ... 8 8 2] [3 5 7 ... 8 5 2]
Accuracy:  0.7565853658536585
------------------------------------------------------------------------------------------------------
Itreation:  20
Loss:  0.7422248172081924
[3 6 7 ... 8 8 2] [3 5 7 ... 8 5 2]
Accuracy:  0.8279512195121951
------------------------------------------------------------------------------------------------------
Itreation:  30
Loss:  0.5890568601530033
[3 6 7 ... 8 8 2] [3 5 7 ... 8 5 2]
Accuracy:  0.8536585365853658
------------------------------------------------------------------------------------------------------
Itreation:  40
Loss:  0.5108159808436484
[3 6 7 ... 8 8 2] [3 5 7 ... 8 5 2]
Accuracy:  0.8678048780487805
-------------------------------------------------------

: 